## PS4 - exam timetable optimization (Hill Climbing)

6 courses, 4 slots, 8 conflicting pairs. state is a list of 6 slot numbers, one per course. cost = 10 x number of conflicting pairs sharing a slot + 2 x courses beyond 2 in any slot.

In [1]:
import sys
import time


def read_input():
    data = sys.stdin.read().split("\n")
    idx = 0
    n, s = map(int, data[idx].split())
    idx += 1
    k = int(data[idx].strip())
    idx += 1
    conflicts = []
    for _ in range(k):
        a, b = map(int, data[idx].split())
        conflicts.append((a, b))
        idx += 1
    initial = list(map(int, data[idx].split()))
    return n, s, conflicts, initial


def calculate_cost(state, conflicts):
    num_conflicts = 0
    for a, b in conflicts:
        if state[a - 1] == state[b - 1]:
            num_conflicts += 1
    conflict_penalty = num_conflicts * 10

    slot_counts = {}
    for slot in state:
        slot_counts[slot] = slot_counts.get(slot, 0) + 1
    excess_courses = sum(max(0, cnt - 2) for cnt in slot_counts.values())
    distribution_penalty = excess_courses * 2

    total = conflict_penalty + distribution_penalty
    return {
        "num_conflicts": num_conflicts,
        "conflict_penalty": conflict_penalty,
        "excess_courses": excess_courses,
        "distribution_penalty": distribution_penalty,
        "total": total,
    }


def generate_neighbors(state, s):
    neighbors = []
    for i in range(len(state)):
        for slot in range(1, s + 1):
            if slot != state[i]:
                new_state = state[:]
                new_state[i] = slot
                neighbors.append(new_state)
    return neighbors


def print_timetable(state):
    for i, slot in enumerate(state):
        print(f"C{i + 1} -> Slot {slot}")


def hill_climb(initial, conflicts, s, log=True):
    start_time = time.time()
    current = initial[:]
    current_cost = calculate_cost(current, conflicts)

    if log:
        print("=" * 40)
        print("Algorithm: Hill Climbing")
        print("Problem: Exam Timetable Optimization")
        print("=" * 40)
        print()
        print("Initial Timetable:")
        print()
        print_timetable(current)
        print()
        print(f"Initial Conflict Cost = {current_cost['conflict_penalty']}")
        print(f"Initial Distribution Cost = {current_cost['distribution_penalty']}")
        print(f"Initial Total Cost = {current_cost['total']}")
        print()
        print("-" * 40)

    iterations = 0
    total_evaluated = 0

    while True:
        neighbors = generate_neighbors(current, s)
        best_neighbor = None
        best_cost = None
        for neighbor in neighbors:
            cost = calculate_cost(neighbor, conflicts)
            total_evaluated += 1
            if best_cost is None or cost["total"] < best_cost["total"]:
                best_neighbor = neighbor
                best_cost = cost

        if best_cost["total"] < current_cost["total"]:
            iterations += 1
            current = best_neighbor
            current_cost = best_cost
            if log:
                print()
                print(f"Iteration {iterations}")
                print()
                print("Best Neighbor Timetable:")
                print()
                print_timetable(current)
                print()
                print(f"Conflict Cost = {current_cost['conflict_penalty']}")
                print(f"Distribution Cost = {current_cost['distribution_penalty']}")
                print(f"Total Cost = {current_cost['total']}")
                print()
                print("-" * 40)
        else:
            break

    exec_time = time.time() - start_time

    if log:
        print()
        print("=" * 40)
        print()
        print("Final Timetable:")
        print()
        print_timetable(current)
        print()
        print(f"Final Conflict Cost = {current_cost['conflict_penalty']}")
        print(f"Final Distribution Cost = {current_cost['distribution_penalty']}")
        print(f"Final Cost = {current_cost['total']}")
        print()
        print(f"Iterations = {iterations}")
        print(f"Total Neighboring States Evaluated = {total_evaluated}")
        print(f"Execution Time = {exec_time:.6f} seconds")
        print()
        print("Termination Reason: Local Optimum")
        print("=" * 40)

    return current, current_cost, iterations, total_evaluated, exec_time




try it on something tiny first, 2 courses that conflict, 2 slots, starting both in slot 1 so there's an obvious conflict to fix.

In [2]:
toy_conflicts = [(1,2)]
print(calculate_cost([1,1], toy_conflicts))
print(generate_neighbors([1,1], 2))

{'num_conflicts': 1, 'conflict_penalty': 10, 'excess_courses': 0, 'distribution_penalty': 0, 'total': 10}
[[2, 1], [1, 2]]


cost 10 (one conflict), 2 neighbors as expected (2 courses x 1 other slot each). run hill climbing on it.

In [3]:
toy_final, toy_cost, toy_iters, toy_eval, _ = hill_climb([1,1], toy_conflicts, 2, log=False)
print(toy_final, toy_cost, toy_iters, toy_eval)

[2, 1] {'num_conflicts': 0, 'conflict_penalty': 0, 'excess_courses': 0, 'distribution_penalty': 0, 'total': 0} 1 4


one move and it's fixed, cost drops to 0 straight away, makes sense since separating 2 conflicting courses into 2 slots is trivial. now the real assignment problem.

hand check the cost function against the assignment's own worked example before trusting it on anything bigger.

state [1,1,2,3,4,2]: C1 and C2 both in slot 1 (conflict), C3 and C6 both in slot 2 (conflict), that's 2 conflicts so conflict penalty = 20. slot counts: slot1 has 2, slot2 has 2, slot3 has 1, slot4 has 1, nothing over 2, so distribution penalty = 0. total should be 20.

In [4]:
conflicts = [(1,2),(1,3),(2,4),(2,5),(3,4),(3,6),(4,5),(5,6)]
print(calculate_cost([1,1,2,3,4,2], conflicts))

{'num_conflicts': 2, 'conflict_penalty': 20, 'excess_courses': 0, 'distribution_penalty': 0, 'total': 20}


matches: 20 total, all from conflicts, 0 from distribution. check neighbor generation matches the assignment's listed example too.

In [5]:
neighbors = generate_neighbors([1,1,2,3,4,2], 4)
print(len(neighbors), 'neighbors')
print(neighbors[:6])

18 neighbors
[[2, 1, 2, 3, 4, 2], [3, 1, 2, 3, 4, 2], [4, 1, 2, 3, 4, 2], [1, 2, 2, 3, 4, 2], [1, 3, 2, 3, 4, 2], [1, 4, 2, 3, 4, 2]]


18 neighbors (6 courses x 3 other slots), first 6 match the listed examples for C1 and C2 exactly. run the actual hill climb on that state.

In [6]:
final, cost, iters, evaluated, t = hill_climb([1,1,2,3,4,2], conflicts, 4, log=False)
print(final, cost, iters, evaluated)

[3, 1, 1, 3, 4, 2] {'num_conflicts': 0, 'conflict_penalty': 0, 'excess_courses': 0, 'distribution_penalty': 0, 'total': 0} 2 54


reaches cost 0 in 2 iterations. now the full script, printing the required format for real, on this same initial state.

In [7]:
import io, sys as _sys
_sys.stdin = io.StringIO('6 4\n8\n1 2\n1 3\n2 4\n2 5\n3 4\n3 6\n4 5\n5 6\n1 1 2 3 4 2')
__name__ = '__main__'
import sys
import time


def read_input():
    data = sys.stdin.read().split("\n")
    idx = 0
    n, s = map(int, data[idx].split())
    idx += 1
    k = int(data[idx].strip())
    idx += 1
    conflicts = []
    for _ in range(k):
        a, b = map(int, data[idx].split())
        conflicts.append((a, b))
        idx += 1
    initial = list(map(int, data[idx].split()))
    return n, s, conflicts, initial


def calculate_cost(state, conflicts):
    num_conflicts = 0
    for a, b in conflicts:
        if state[a - 1] == state[b - 1]:
            num_conflicts += 1
    conflict_penalty = num_conflicts * 10

    slot_counts = {}
    for slot in state:
        slot_counts[slot] = slot_counts.get(slot, 0) + 1
    excess_courses = sum(max(0, cnt - 2) for cnt in slot_counts.values())
    distribution_penalty = excess_courses * 2

    total = conflict_penalty + distribution_penalty
    return {
        "num_conflicts": num_conflicts,
        "conflict_penalty": conflict_penalty,
        "excess_courses": excess_courses,
        "distribution_penalty": distribution_penalty,
        "total": total,
    }


def generate_neighbors(state, s):
    neighbors = []
    for i in range(len(state)):
        for slot in range(1, s + 1):
            if slot != state[i]:
                new_state = state[:]
                new_state[i] = slot
                neighbors.append(new_state)
    return neighbors


def print_timetable(state):
    for i, slot in enumerate(state):
        print(f"C{i + 1} -> Slot {slot}")


def hill_climb(initial, conflicts, s, log=True):
    start_time = time.time()
    current = initial[:]
    current_cost = calculate_cost(current, conflicts)

    if log:
        print("=" * 40)
        print("Algorithm: Hill Climbing")
        print("Problem: Exam Timetable Optimization")
        print("=" * 40)
        print()
        print("Initial Timetable:")
        print()
        print_timetable(current)
        print()
        print(f"Initial Conflict Cost = {current_cost['conflict_penalty']}")
        print(f"Initial Distribution Cost = {current_cost['distribution_penalty']}")
        print(f"Initial Total Cost = {current_cost['total']}")
        print()
        print("-" * 40)

    iterations = 0
    total_evaluated = 0

    while True:
        neighbors = generate_neighbors(current, s)
        best_neighbor = None
        best_cost = None
        for neighbor in neighbors:
            cost = calculate_cost(neighbor, conflicts)
            total_evaluated += 1
            if best_cost is None or cost["total"] < best_cost["total"]:
                best_neighbor = neighbor
                best_cost = cost

        if best_cost["total"] < current_cost["total"]:
            iterations += 1
            current = best_neighbor
            current_cost = best_cost
            if log:
                print()
                print(f"Iteration {iterations}")
                print()
                print("Best Neighbor Timetable:")
                print()
                print_timetable(current)
                print()
                print(f"Conflict Cost = {current_cost['conflict_penalty']}")
                print(f"Distribution Cost = {current_cost['distribution_penalty']}")
                print(f"Total Cost = {current_cost['total']}")
                print()
                print("-" * 40)
        else:
            break

    exec_time = time.time() - start_time

    if log:
        print()
        print("=" * 40)
        print()
        print("Final Timetable:")
        print()
        print_timetable(current)
        print()
        print(f"Final Conflict Cost = {current_cost['conflict_penalty']}")
        print(f"Final Distribution Cost = {current_cost['distribution_penalty']}")
        print(f"Final Cost = {current_cost['total']}")
        print()
        print(f"Iterations = {iterations}")
        print(f"Total Neighboring States Evaluated = {total_evaluated}")
        print(f"Execution Time = {exec_time:.6f} seconds")
        print()
        print("Termination Reason: Local Optimum")
        print("=" * 40)

    return current, current_cost, iterations, total_evaluated, exec_time


def main():
    n, s, conflicts, initial = read_input()
    hill_climb(initial, conflicts, s)


if __name__ == "__main__":
    main()


Algorithm: Hill Climbing
Problem: Exam Timetable Optimization

Initial Timetable:

C1 -> Slot 1
C2 -> Slot 1
C3 -> Slot 2
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Initial Conflict Cost = 20
Initial Distribution Cost = 0
Initial Total Cost = 20

----------------------------------------

Iteration 1

Best Neighbor Timetable:

C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 2
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Conflict Cost = 10
Distribution Cost = 0
Total Cost = 10

----------------------------------------

Iteration 2

Best Neighbor Timetable:

C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 1
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Conflict Cost = 0
Distribution Cost = 0
Total Cost = 0

----------------------------------------


Final Timetable:

C1 -> Slot 3
C2 -> Slot 1
C3 -> Slot 1
C4 -> Slot 3
C5 -> Slot 4
C6 -> Slot 2

Final Conflict Cost = 0
Final Distribution Cost = 0
Final Cost = 0

Iterations = 2
Total Neighboring States Evaluated = 54
Execution Time = 0.000000 seconds

Termination Reason: Loca

now the 3 required experiments with the assignment's 3 given initial timetables, comparing results.

In [8]:
experiments = {
    1: [1,1,2,3,4,2],
    2: [2,3,1,1,4,4],
    3: [4,2,3,1,2,3],
}
results = {}
for num, init in experiments.items():
    final, cost, iters, evaluated, t = hill_climb(init, conflicts, 4, log=False)
    init_cost = calculate_cost(init, conflicts)
    results[num] = (init, init_cost['total'], final, cost['total'], iters, evaluated, t)
    print(f'Experiment {num}: initial cost {init_cost["total"]} -> final {final}, cost {cost["total"]}, {iters} iterations, {evaluated} neighbors evaluated')

Experiment 1: initial cost 20 -> final [3, 1, 1, 3, 4, 2], cost 0, 2 iterations, 54 neighbors evaluated
Experiment 2: initial cost 20 -> final [2, 3, 3, 1, 2, 4], cost 0, 2 iterations, 54 neighbors evaluated
Experiment 3: initial cost 20 -> final [4, 2, 2, 1, 4, 3], cost 0, 2 iterations, 54 neighbors evaluated


all three starting timetables land on total cost 0, just at different final assignments. checking if this conflict graph is just easy to satisfy: max degree in the conflict graph is 3 (courses 2, 3, 4, and 5 each conflict with 3 others), well under the 4 available slots, so a clean split should exist, and hill climbing found one from every starting point tried here.